# Traitement des données 

In [80]:
import pandas as pd
import datetime

In [81]:
data = pd.read_csv('LECLERC_cleaned.csv')

In [82]:
data = data.loc[data['Product Name'] != 'Non trouvé']

## Traitement de la date de livraison

Les données de livraison ne sont pas directement traitables car les données scrappées retournent des intervalles entre deux dates le plus souvent et séparées par des mots: "Prévue entre le ... et le ...". 

Nous supprimons les mots et ne gardons que la date la plus élevée. 

Le code ci-dessous retourne la différence de jours entre la date du scrapping et la date de livraison. 

In [83]:
def calculate_days(row):
    date_range = row['Delivery Date']
    date_range = date_range.replace('Prévue entre le ', '')
    date_range = date_range.replace('Prévue le ', '')
    if ' et le ' in date_range:
        max_date = date_range.split(' et le ')[-1]
    else:
        max_date = date_range
    max_date_dt = datetime.datetime.strptime(max_date, '%d/%m/%y')
    scrap_date = datetime.datetime.strptime(row['Timestamp'], '%d/%m/%Y %H:%M:%S')
    scrap_date = datetime.datetime.combine(scrap_date.date(), datetime.time(0, 0))
    
    difference = (max_date_dt - scrap_date).days
    
    return difference

Le code ci-dessous servira plus tard après l'import de données sur les vendeurs.

In [84]:
import numpy as np

def calculate_experience(row):
    if not pd.isna(row['SellerActivityDate']):
        start = str(row['SellerActivityDate'])
        start_dt = datetime.datetime.strptime(start, '%d/%m/%Y')
        try:
            scrap_date = datetime.datetime.strptime(row['Timestamp'], '%d/%m/%Y %H:%M:%S')
        except ValueError as e:
            print(f"Error parsing timestamp: {row['Timestamp']}")
            raise e
        
        scrap_date = scrap_date.date()
        start_dt = start_dt.date()
        
        difference = (scrap_date - start_dt).days
    else:
        difference = np.nan
    
    return difference 



In [85]:
data['shipping_days'] = data.apply(calculate_days, axis=1)

## Ajout des informations vendeurs 

In [86]:
data.drop(columns = ['Seller Rating','Delivery Date','Platform'],inplace=True)

Nous avons recupéré manuellement ces informations sur les vendeurs depuis le site de Leclerc:
- la date depuis laquelle il exerce sur le site
- sa note 
- le nombre de notes qu'il a obtenues
-  son pays d'exercice


Nous traitons ces données puis les rajoutons aux données scrappées


In [87]:
seller = pd.read_csv('Sellers.csv')
seller.drop(columns = ['SellerStatus'],inplace=True)

In [88]:
data_merge = pd.merge(data,seller,how='left',on='Seller')

In [89]:
data_merge.isna().sum()/len(data_merge)

Product Name          0.000000
Seller                0.000000
Price                 0.000000
Delivery Fees         0.000000
Product State         0.000000
Timestamp             0.000000
ID                    0.000000
shipping_days         0.000000
Seller Rating         0.101317
NbSellerRatings       0.101317
SellerActivityDate    0.101317
SellerCountry         0.000000
dtype: float64

In [90]:
data_merge[data_merge['SellerActivityDate'].apply(lambda x: isinstance(x, float))].head(3)

,Product Name,Seller,Price,Delivery Fees,Product State,Timestamp,ID,shipping_days,Seller Rating,NbSellerRatings,SellerActivityDate,SellerCountry
0,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",E.Leclerc,1349.0,Offerte,NEUF,24/12/2024 13:30:11,dbbfae24-b0a8-4035-8b89-4e0ee835b93e,4,NaN,NaN,NaN,France
9,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",E.Leclerc,1098.0,Offerte,NEUF,24/12/2024 13:30:17,d5ccaf94-fda7-4de4-a3b8-a9b219fe70ed,4,NaN,NaN,NaN,France
27,"Apple iPhone 16 Plus 17 cm (6.7"") Double SIM i...",E.Leclerc,1240.0,Offerte,NEUF,24/12/2024 13:30:38,ed249810-8dce-4cc4-a29a-28cf62130b9f,4,NaN,NaN,NaN,France


Nous traitons la date depuis laquelle le vendeur exerce et la transformons en nombre de jours d'experience en faisant la difference entre cette date et la date du scrapping.

In [91]:
data_merge['SellerExperience'] = data_merge.apply(calculate_experience, axis=1)

Suppression de delivery fees car tous les frais de livraison sont nuls, probablement car l'iPhone est un bien cher. 

In [92]:
columns_to_delete = ['Delivery Fees','SellerActivityDate','Product State']

## Traitement des variables qualitatives

Nous transformons les variables qualitatives à m modalités en m-1 variables quantitatives binaires avec la variable la plus fréquente comme variable de référence

In [93]:
#Supprimer la catégorie d'état de téléphone la plus fréquente comme variable de reférence
len(data_merge[data_merge['Product State'] == 'NEUF']['ID'].unique())/len(data_merge['ID'].unique())

0.9172413793103448

In [94]:
dummies_state = pd.get_dummies(data_merge['Product State'], drop_first=True)
data_merge = pd.concat([data_merge, dummies_state], axis=1)

In [95]:
data_merge['SellerCountry'].unique()

array(['France', 'Espagne', 'Luxembourg', 'Lettonie', 'Italie',
       'France       '], dtype=object)

In [96]:
data_merge['SellerCountry'].replace({
    'France       ':'France'
},inplace=True)

C:\Users\zoero\AppData\Local\Temp\ipykernel_13744\2858434610.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_merge['SellerCountry'].replace({


In [97]:
dummies_country = pd.get_dummies(data_merge['SellerCountry'])
dummies_country.drop('France', axis=1, inplace=True)
data_merge = pd.concat([data_merge, dummies_country], axis=1)

In [98]:
data_merge.head()

,Product Name,Seller,Price,Delivery Fees,Product State,Timestamp,ID,shipping_days,Seller Rating,NbSellerRatings,...,SellerExperience,OCCASION - BON ÉTAT,OCCASION - EXCELLENT ÉTAT,OCCASION - PARFAIT - JAMAIS UTILISÉ,OCCASION - TRÉS BON ÉTAT,OCCASION - ÉTAT CORRECT,Espagne,Italie,Lettonie,Luxembourg
0,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",E.Leclerc,1349.00,Offerte,NEUF,24/12/2024 13:30:11,dbbfae24-b0a8-4035-8b89-4e0ee835b93e,4,NaN,NaN,...,NaN,False,False,False,False,False,False,False,False,False
1,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Icoza,1408.17,Offerte,NEUF,24/12/2024 13:30:11,9b753e1c-cafe-4c0c-9882-e8e7d4539f4a,4,4.0,140.0,...,1565.0,False,False,False,False,False,False,False,False,False
2,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Stock e-commerce,1409.17,Offerte,NEUF,24/12/2024 13:30:11,adf44090-f1c1-4f2a-95e5-7f95d7035c54,4,4.0,242.0,...,1286.0,False,False,False,False,False,False,False,False,False
3,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Maxmovil,1432.28,Offerte,NEUF,24/12/2024 13:30:11,eafc127f-641f-41db-8dfe-eacddf263a9d,4,4.0,14.0,...,1278.0,False,False,False,False,False,True,False,False,False
4,"Apple iPhone 16 15,5 cm (6.1"") Double SIM iOS ...",Monsieurplus,1473.99,Offerte,NEUF,24/12/2024 13:30:11,85c9fb64-cc14-4ab9-a98d-444cf8373ea1,4,4.0,7.0,...,735.0,False,False,False,False,False,False,False,False,False


In [99]:
data_merge.drop(columns = columns_to_delete,inplace=True)

In [100]:
data_merge.isna().sum()

Product Name                               0
Seller                                     0
Price                                      0
Timestamp                                  0
ID                                         0
shipping_days                              0
Seller Rating                          20622
NbSellerRatings                        20622
SellerCountry                              0
SellerExperience                       20622
OCCASION - BON ÉTAT                        0
OCCASION - EXCELLENT ÉTAT                  0
OCCASION - PARFAIT - JAMAIS UTILISÉ        0
OCCASION - TRÉS BON ÉTAT                   0
OCCASION - ÉTAT CORRECT                    0
Espagne                                    0
Italie                                     0
Lettonie                                   0
Luxembourg                                 0
dtype: int64

## Note et Nombre de notes de Leclerc 

Ne pouvant obtenir de notes pour le vendeur Leclerc nous reprenons la méthode utilisée dans l'article pour ce cas de figure lui donnons la note moyenne et le nombre de notes moyen. 

In [101]:
mean_nb_rating = data_merge['NbSellerRatings'].mean()
mean_nb_rating

np.float64(79.60978476576808)

In [102]:
median_rating = data_merge['Seller Rating'].mean()
data_merge['Seller Rating'].fillna(median_rating, inplace=True)
data_merge['NbSellerRatings'].fillna(mean_nb_rating, inplace=True)

C:\Users\zoero\AppData\Local\Temp\ipykernel_13744\1743373068.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_merge['Seller Rating'].fillna(median_rating, inplace=True)
C:\Users\zoero\AppData\Local\Temp\ipykernel_13744\1743373068.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a

In [103]:
median_seller_experience = data_merge['SellerExperience'].median()
data_merge['SellerExperience'].fillna(median_seller_experience,inplace=True)

C:\Users\zoero\AppData\Local\Temp\ipykernel_13744\1889923872.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_merge['SellerExperience'].fillna(median_seller_experience,inplace=True)


In [104]:
data_merge.drop(columns=['SellerCountry'],inplace=True)

In [105]:
data_merge.isna().sum()

Product Name                           0
Seller                                 0
Price                                  0
Timestamp                              0
ID                                     0
shipping_days                          0
Seller Rating                          0
NbSellerRatings                        0
SellerExperience                       0
OCCASION - BON ÉTAT                    0
OCCASION - EXCELLENT ÉTAT              0
OCCASION - PARFAIT - JAMAIS UTILISÉ    0
OCCASION - TRÉS BON ÉTAT               0
OCCASION - ÉTAT CORRECT                0
Espagne                                0
Italie                                 0
Lettonie                               0
Luxembourg                             0
dtype: int64

In [106]:
boolean_variables = [                         
'OCCASION - BON ÉTAT',                    
'OCCASION - EXCELLENT ÉTAT',              
'OCCASION - PARFAIT - JAMAIS UTILISÉ',    
'OCCASION - TRÉS BON ÉTAT',               
'OCCASION - ÉTAT CORRECT' ,               
'Espagne'        ,                        
'Italie'        ,                         
'Lettonie' ,                              
'Luxembourg']

In [107]:
data_merge.dtypes

Product Name                            object
Seller                                  object
Price                                  float64
Timestamp                               object
ID                                      object
shipping_days                            int64
Seller Rating                          float64
NbSellerRatings                        float64
SellerExperience                       float64
OCCASION - BON ÉTAT                       bool
OCCASION - EXCELLENT ÉTAT                 bool
OCCASION - PARFAIT - JAMAIS UTILISÉ       bool
OCCASION - TRÉS BON ÉTAT                  bool
OCCASION - ÉTAT CORRECT                   bool
Espagne                                   bool
Italie                                    bool
Lettonie                                  bool
Luxembourg                                bool
dtype: object

In [108]:
for c in boolean_variables:
    data_merge[c].replace({
        True:1,
        False:0
    },inplace=True)

C:\Users\zoero\AppData\Local\Temp\ipykernel_13744\3291783102.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_merge[c].replace({
C:\Users\zoero\AppData\Local\Temp\ipykernel_13744\3291783102.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_merge[c].replace({


## Suppression des données produit qui s'étendent sur moins de 3 jours

Nous conservons les produits qui étaient disponibles à la vente de manière stable pendant trois jours consécutifs (c'est à dire qui apparaissent au moins une fois par jour dans nos données).

In [109]:
from datetime import timedelta

# Trier les données
df = data_merge.sort_values(by=['ID', 'Timestamp'])
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%d/%m/%Y %H:%M:%S')

# Filtrer les données à partir du 26/12
start_date = pd.to_datetime('26/12/2024')
df = df[df['Timestamp'] >= start_date]

# Trier les données
df = df.sort_values(by=['ID', 'Timestamp'])

def filter_and_identify_sequences(group):
    group = group.sort_values(by='Timestamp')
    group['time_diff'] = group['Timestamp'].diff().fillna(pd.Timedelta(seconds=0))
    breaks = group['time_diff'] > timedelta(hours=24)
    group['block'] = breaks.cumsum()

    valid_blocks = group.groupby('block').filter(lambda x: (x['Timestamp'].max() - x['Timestamp'].min()) >= timedelta(days=2))
    non_valid_blocks = group[~group['block'].isin(valid_blocks['block'].unique())]

    return valid_blocks, non_valid_blocks

results = df.groupby('ID').apply(lambda g: filter_and_identify_sequences(g))

valid_data = pd.concat([result[0] for result in results])
non_valid_data = pd.concat([result[1] for result in results])


C:\Users\zoero\AppData\Local\Temp\ipykernel_13744\2499999839.py:8: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  start_date = pd.to_datetime('26/12/2024')
C:\Users\zoero\AppData\Local\Temp\ipykernel_13744\2499999839.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  results = df.groupby('ID').apply(lambda g: filter_and_identify_sequences(g))


In [110]:
len(valid_data)

201222

In [111]:
len(data_merge)

203539

In [112]:
len(non_valid_data)

2223

In [113]:
data_merge['Timestamp'] = pd.to_datetime(data_merge['Timestamp'], format='%d/%m/%Y %H:%M:%S')
data_merge['key'] = data_merge['ID'].astype(str) + data_merge['Timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')
non_valid_data['key'] = non_valid_data['ID'].astype(str) + non_valid_data['Timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

keys_to_remove = set(non_valid_data['key'])
len(keys_to_remove)
data_merge['to_remove'] = data_merge['key'].isin(keys_to_remove)

cleaned_df = data_merge[~data_merge['to_remove']].drop(columns=['key', 'to_remove'])  # Supprimer les colonnes temporaires


In [114]:
cleaned_df['Leclerc'] = cleaned_df['Seller'].apply(lambda x: 1 if x == 'E.Leclerc' else 0)

Export des données

In [115]:
cleaned_df.to_csv('LECLERC_ordered.csv')